# 08 — AI-Ready — ALL MODELS Results & Evidence Insights (v2.1) (PyCIL)

**Tujuan:** mengubah hasil **production** EFCIL × Attention menjadi *evidence pack* untuk dianalisis secara transparan oleh AI (CSV + JSON + Markdown + ZIP). Notebook ini adalah **pelengkap**, bukan pengganti `07_results_analysis.ipynb`.

- Membaca hasil dari **PC2** dan ekspor **server RTX A5000**.
- Menggunakan **canonical predictions** (FeTrIL=SVM, FeCAM=Mahalanobis, SSRE/EFC/PASS sesuai runner) untuk audit, bukan accuracy smoke test.
- Mengeluarkan data untuk semua metode, backbone, protocol, dan attention yang tersedia.
- Memisahkan observasi numerik yang tervalidasi dari interpretasi sebab-akibat yang *harus* diperiksa kembali.
- **Tidak mengirim data ke layanan AI** dan tidak membutuhkan API key. Setelah dihasilkan, upload ZIP laporan ke ChatGPT/AI yang Anda pilih.

> Tidak menyertakan dataset satelit atau checkpoint `.pkl` di ZIP final. `ready_snapshot` berarti inventory checkpoint ada pada mesin sumber, bukan weights berhasil dimuat.


## 1 — Prasyarat & lokasi file

Simpan dua file dalam proyek yang sama:

```text
PyCIL/
├── notebooks/08_ai_ready_insights.ipynb
├── scripts/ai_ready_results_export.py
├── outputs/experiments/<run_id>/...            # produksi PC2
└── analysis_inputs/server/experiments/<run_id>/... # hasil ZIP server yang telah diekstrak
```

Ekspor server dapat dibuat dengan `python scripts/export_evaluation_artifacts.py --machine server`, lalu pindahkan dan ekstrak ke `analysis_inputs/server/` di PC2. Gunakan kernel Python yang memiliki `numpy` dan `pandas`. Ekspor hanya saat tidak ada file output training yang sedang ditulis.

In [ ]:
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
ENGINE = PROJECT_ROOT / "scripts" / "ai_ready_results_export.py"
if not ENGINE.is_file():
    raise FileNotFoundError(f"Script tidak ditemukan: {ENGINE}. Simpan script ke PyCIL/scripts/.")
spec = importlib.util.spec_from_file_location("ai_ready_results_export", ENGINE)
ai_exporter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ai_exporter)
EXPECTED_EXPORTER_VERSION = "2.1.0"
EXPORTER_VERSION = getattr(ai_exporter, "AI_READY_EXPORTER_VERSION", None)
if EXPORTER_VERSION != EXPECTED_EXPORTER_VERSION:
    raise RuntimeError(
        "VERSI EXPORTER TIDAK COCOK. Notebook memerlukan "
        f"ai_ready_results_export.py v{EXPECTED_EXPORTER_VERSION}, "
        f"tetapi ditemukan versi {EXPORTER_VERSION or 'lama/tidak dikenal'} di: {ENGINE}\n"
        "Ganti file PyCIL/scripts/ai_ready_results_export.py dengan versi dari paket v2.1, "
        "kemudian jalankan ulang notebook dari Cell 2 (atau Restart Kernel & Run All)."
    )
print("PyCIL root:", PROJECT_ROOT)
print("Exporter:", ENGINE, "| version:", EXPORTER_VERSION)
print("pandas:", pd.__version__, "numpy:", np.__version__)


## 2 — Konfigurasi sumber hasil & cakupan

Pengaturan default mengikuti hasil AID production (80 konfigurasi), seed 1993. Ganti `DATASET_SCOPE` menjadi `"all"` **hanya jika** hasil NWPU sudah tersedia. Jika nama kelas tidak dapat dipetakan secara benar dari `splits/.../train.csv`, buat CSV manual `analysis_inputs/class_labels.csv` dengan kolom `dataset,incremental_id,class_name`. Jangan menebak urutan kelas.

In [ ]:
DATASET_SCOPE = "aid"      # "aid" | "nwpu" | "all"
TRAINING_SEED = 1993
SOURCES = {
    "pc2": PROJECT_ROOT / "outputs" / "experiments",
    "server": PROJECT_ROOT / "analysis_inputs" / "server" / "experiments",
}
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analysis_ai_ready"
LABELS_FILE = PROJECT_ROOT / "analysis_inputs" / "class_labels.csv"
LABELS_FILE = LABELS_FILE if LABELS_FILE.exists() else None

for machine, path in SOURCES.items():
    print(machine, "=>", path, "| exists:", path.is_dir())
print("Labels override:", LABELS_FILE or "none; read from manifest if possible, otherwise Cxx")
print("Output directory:", OUTPUT_DIR)


## 3 — Audit, perhitungan, insight otomatis, dan ZIP

Cell berikut memverifikasi kelengkapan run, kecocokan *confusion matrix* vs `y_true`/`y_pred`, konsistensi AIA/LTA, checkpoint manifest/inventory, serta fairness pairing sebelum menghitung delta. Hasil partial **tidak** dianggap valid.

In [ ]:
if getattr(ai_exporter, "AI_READY_EXPORTER_VERSION", None) != EXPECTED_EXPORTER_VERSION:
    raise RuntimeError("Objek exporter di kernel masih versi lama. Jalankan ulang Cell 2 atau Restart Kernel & Run All.")
result = ai_exporter.main(
    project_root=PROJECT_ROOT,
    sources=SOURCES,
    output_dir=OUTPUT_DIR,
    dataset_scope=DATASET_SCOPE,
    seed=TRAINING_SEED,
    class_labels_file=LABELS_FILE,
    create_zip=True,
    allow_inventory=True,
)
if result["manifest"].get("schema_version") != EXPECTED_EXPORTER_VERSION:
    raise RuntimeError("Versi manifest tidak sesuai notebook: " + str(result["manifest"].get("schema_version")))
REQUIRED_FILES = list(ai_exporter.REQUIRED_ALL_MODELS_OUTPUTS)
MISSING = [n for n in REQUIRED_FILES if not (OUTPUT_DIR / n).is_file()]
if MISSING:
    raise RuntimeError(
        "EXPORT TIDAK LENGKAP. Missing: " + ", ".join(MISSING)
        + ". Periksa status Cell 6 dan pastikan scripts/ai_ready_results_export.py v2.1."
    )
print("VALIDASI OUTPUT ALL MODELS: PASSED")
for n in REQUIRED_FILES:
    print("  OK:", n, f"({(OUTPUT_DIR / n).stat().st_size:,} bytes)")
print(json.dumps(result["manifest"], indent=2, ensure_ascii=False))
print("\nAI-READY ZIP:", result["zip_path"])


## 4 — Status, coverage, dan daftar run yang perlu diperbaiki

`expected_coverage.csv` memiliki **80 baris untuk AID**: 5 metode × 2 backbone × 2 protocol × 4 attention. Jika belum 80/80, insight hanya bersifat sementara.

In [ ]:
audit = pd.read_csv(OUTPUT_DIR / "run_audit.csv")
coverage = pd.read_csv(OUTPUT_DIR / "expected_coverage.csv")
valid_runs = pd.read_csv(OUTPUT_DIR / "valid_runs.csv")
print("Valid", int(coverage.ready.sum()), "/", len(coverage))
if not audit.empty:
    display(audit.groupby("status", dropna=False).size().rename("n_runs").reset_index())
    bad = audit[~audit.status.isin(["ready_full", "ready_snapshot", "duplicate_copy"])]
    if not bad.empty:
        display(bad[["run_id", "source_machine", "status", "issue"]].head(20))
if not valid_runs.empty:
    display(valid_runs[["dataset","method","protocol","backbone","attention","aia","lta","status","run_id"]].head(12))


## 5 — Attention Gain dan fairness check

- **ΔAIA**: AIA dengan attention − AIA baseline None (pp).
- **Mean ΔAIA**: rata-rata delta dari 4 pasangan `R18/R34 × T5/T10` per metode **jika lengkap**.
- `n_pairs=4` bukan empat seed independen. Pasangan yang machine/batch/histogram evaluasinya berbeda akan ditolak.

In [ ]:
paired = pd.read_csv(OUTPUT_DIR / "paired_attention_effects.csv")
gain = pd.read_csv(OUTPUT_DIR / "method_attention_gain.csv")
rejected = pd.read_csv(OUTPUT_DIR / "rejected_pairs.csv")
if not gain.empty:
    display(gain[["method","attention","n_pairs","expected_pairs","complete_bundle","mean_delta_aia_pp","mean_delta_lta_pp","win_rate_pct"]].sort_values("mean_delta_aia_pp",ascending=False).round(3))
if not rejected.empty:
    display(Markdown("**Pasangan yang ditolak untuk perbandingan attention:**"))
    display(rejected)
if not paired.empty:
    display(paired[["method","protocol","backbone","attention","delta_aia_pp","delta_lta_pp","baseline_run_id","attention_run_id"]].head(20).round(3))


## 6 — Class accuracy: semua attention & tiap task

`class_accuracy_pct` di sini adalah **recall per kelas** (`TP / jumlah sampel sebenarnya dari kelas tersebut`). Bandingkan keempat attention setelah mengecek `class_label_verified` dan jumlah sampel `support`. Kelas yang belum diketahui namanya tetap ditampilkan sebagai `C00`, `C01`, dan seterusnya.

In [ ]:
CLASS_METHOD = "pass"     # fetril | fecam | ssre | efc | pass
CLASS_PROTOCOL = "T5"
CLASS_BACKBONE = "r18"

wide = pd.read_csv(OUTPUT_DIR / "class_attention_final_wide.csv")
class_long = pd.read_csv(OUTPUT_DIR / "class_accuracy_long.csv")
class_delta = pd.read_csv(OUTPUT_DIR / "class_attention_delta.csv")
selection = wide[(wide.method == CLASS_METHOD)&(wide.protocol == CLASS_PROTOCOL)&
                 (wide.backbone == CLASS_BACKBONE)] if not wide.empty else wide
if selection.empty:
    print("Model yang dipilih belum memiliki hasil class accuracy.")
else:
    cols=[c for c in ["class_id","class_name","support","none","eca","cbam","procbam","delta_eca_pp","delta_cbam_pp","delta_procbam_pp"] if c in selection]
    display(selection[cols].round(2).head(45))
    if not class_delta.empty:
        tmp=class_delta[(class_delta.method==CLASS_METHOD)&(class_delta.protocol==CLASS_PROTOCOL)&
                        (class_delta.backbone==CLASS_BACKBONE)].copy()
        if not tmp.empty:
            display(tmp[["task_id","class_id","class_name","attention","baseline_class_accuracy_pct","attention_class_accuracy_pct","delta_class_accuracy_pp","attention_support"]].sort_values("delta_class_accuracy_pp").head(10).round(2))


## 7 — Per-class forgetting, old/new accuracy, dan perubahan confusion matrix

Dua pertanyaan berbeda: apakah accuracy meningkat pada kelas baru (*plasticity*) dan apakah akurasi kelas lama bertahan (*stability*)? Jangan menyimpulkan attention lebih baik hanya dari satu kelas. Tinjau `task_attention_delta.csv`, `class_forgetting_attention_delta.csv`, dan `misclassification_delta.csv`.

In [ ]:
task_delta = pd.read_csv(OUTPUT_DIR / "task_attention_delta.csv")
forget_delta = pd.read_csv(OUTPUT_DIR / "class_forgetting_attention_delta.csv")
mis_delta = pd.read_csv(OUTPUT_DIR / "misclassification_delta.csv")
if not task_delta.empty:
    print("Task-level old/new accuracy delta:")
    display(task_delta[["method","protocol","backbone","attention","task_id","delta_top1_pp","delta_old_pp","delta_new_pp"]].head(20).round(2))
if not forget_delta.empty:
    print("Per-class forgetting delta (negative = less forgetting):")
    display(forget_delta.sort_values("delta_forgetting_pp").head(12).round(2))
if not mis_delta.empty:
    print("Misclassification directions most reduced:")
    display(mis_delta.sort_values("errors_reduced",ascending=False).head(12)[["method","protocol","backbone","attention","true_class","predicted_class","support","errors_none","errors_attention","errors_reduced"]])


## 8 — Highlight otomatis dan laporan lengkap seluruh model yang dapat diaudit

Insight dibuat dengan aturan deterministik (tanpa LLM). Setiap insight menyertakan **angka dan file bukti**. Anda atau AI tetap perlu mengecek konsistensi, keterbatasan sampel, dan mekanisme ilmiahnya.
**Penting:** `insights.json` adalah highlight terpilih, bukan cakupan seluruh model. Analisis lengkap ada pada `insights_all.json`, `model_insights_all.csv`, `bundle_insights_all.csv`, `attention_insights_all.csv`, dan `class_insights_final_all.csv`.


In [ ]:
insights_doc = json.loads((OUTPUT_DIR / "insights.json").read_text(encoding="utf-8"))
print("Jumlah temuan:", insights_doc["count"])
for item in insights_doc["insights"][:12]:
    evidence = ", ".join(e["file"] for e in item["evidence"])
    display(Markdown(f"**{item['insight_id']} — {item['title']}**\n\n{item['claim']}\n\n*Bukti: {evidence}*"))
print("Seluruh temuan ada di insights.json dan AI_EVIDENCE_REPORT.md")


## 9 — Verifikasi cakupan: SEMUA 80 model, 20 bundle, dan 60 pasangan AID

Bagian ini **wajib** untuk memastikan tidak ada konfigurasi yang hanya terwakili oleh contoh *top-k*. File `model_insights_all.csv` mencakup baseline None juga. File `attention_insights_all.csv` mencakup semua pasangan dengan status `paired_valid` atau alasan belum tersedia. Jika hasil production AID selesai 80/80, harus ada **80 model, 20 bundle, 60 pasangan valid**, dan **1.800 rekaman kelas final** (60 pasangan × 30 kelas). Tidak ada analisis yang diisi dengan angka palsu ketika output belum selesai.


In [ ]:
all_models = pd.read_csv(OUTPUT_DIR / "model_insights_all.csv")
all_bundles = pd.read_csv(OUTPUT_DIR / "bundle_insights_all.csv")
all_pairs = pd.read_csv(OUTPUT_DIR / "attention_insights_all.csv")
all_classes_final = pd.read_csv(OUTPUT_DIR / "class_insights_final_all.csv")

expected_models = 80 if DATASET_SCOPE == "aid" else (160 if DATASET_SCOPE == "all" else 80)
expected_pairs = expected_models // 4 * 3
print(f"Expected model configurations: {expected_models} | Exported: {len(all_models)}")
print(f"Expected comparison bundles: {expected_models//4} | Exported: {len(all_bundles)}")
print(f"Expected attention pairs: {expected_pairs} | Exported: {len(all_pairs)}")
print(f"Fair pairs: {(all_pairs.status == 'paired_valid').sum()} | Other statuses: {(all_pairs.status != 'paired_valid').sum()}")
print(f"Final per-class comparisons: {len(all_classes_final)}")
assert len(all_models) == expected_models, "Some model configurations were omitted"
assert len(all_bundles) == expected_models // 4, "Some bundles were omitted"
assert len(all_pairs) == expected_pairs, "Some attention comparisons were omitted"
if all_models.ready.all():
    assert all_pairs.status.eq('paired_valid').all(), "All models ready but some comparisons failed fairness"
    expected_class_rows = 60*30 if DATASET_SCOPE == 'aid' else (60*45 if DATASET_SCOPE == 'nwpu' else 60*(30+45))
    assert len(all_classes_final) == expected_class_rows, "Some final per-class comparisons were omitted"
    print("FULL COVERAGE VALIDATION: PASSED")
else:
    print("Coverage partial; missing model/pair rows remain explicitly flagged")

print("\nFull model summary (no .head() cutoff):")
with pd.option_context('display.max_rows', 200, 'display.max_columns', 14):
    display(all_models[['dataset','method','protocol','backbone','attention','status','aia_pct','lta_pct','macro_f1_pct','run_id']])


## 10 — Semua perbandingan per-model + insight per kelas

Tabel berikut menampilkan **seluruh 60 perbandingan AID** (atau 120 ketika kedua dataset dipilih). Masing-masing baris menjelaskan ΔAIA/LTA dan jumlah kelas yang meningkat/menurun/tetap, bukan sekadar lima kasus terbesar. Untuk angka setiap kelas dan task, AI harus membaca `class_insights_final_all.csv`, `class_attention_delta.csv`, dan `class_accuracy_long.csv`.


In [ ]:
cols = [
    'dataset','method','protocol','backbone','attention','status',
    'delta_aia_pp','delta_lta_pp','delta_macro_f1_pp','delta_fr_pp',
    'classes_improved','classes_degraded','classes_unchanged',
    'best_class_name','best_class_delta_pp',
    'worst_class_name','worst_class_delta_pp',
]
cols = [c for c in cols if c in all_pairs]
with pd.option_context('display.max_rows', 200, 'display.max_columns', 22):
    display(all_pairs[cols].round(3))

print("\nCoverage per metode dan backbone:")
with pd.option_context('display.max_rows', 60):
    display(all_pairs.groupby(['dataset','method','backbone','protocol','status']).size().rename('pairs').reset_index())

print("\nClass accuracy detail (file contains all class rows):")
print("class_insights_final_all.csv rows:",len(all_classes_final))
if not all_classes_final.empty:
    display(all_classes_final[['method','protocol','backbone','attention','class_id','class_name',
                               'baseline_class_accuracy_pct','attention_class_accuracy_pct',
                               'delta_class_accuracy_pp']].head(12).round(2))
print("\nFull report:", OUTPUT_DIR / 'AI_EVIDENCE_REPORT.md')
print("Full JSON:", OUTPUT_DIR / 'insights_all.json')


## 11 — Unggah evidence pack ke AI

File yang siap dibaca AI adalah **`outputs/analysis_ai_ready/ai_ready_evaluation_bundle.zip`**. Bundle berisi CSV per run/per class/per task, statistik attention, report Markdown, insight JSON, kamus data, dan prompt analisis.

Saat ingin menganalisis Bab IV, unggah ZIP ke ChatGPT/AI bersama prompt yang otomatis disimpan pada `AI_PROMPT.md`. File ini **tidak** menyertakan citra dataset, raw individual predictions, atau model weights. Hal itu menjaga ukuran relatif kecil dan memudahkan AI fokus pada evidence terstruktur.

**Penting:** jika hasil PC/server belum semuanya diekspor, output final akan bersifat *preliminary*. Jalankan ulang setelah semua production, recovery, dan ekspor server selesai.

In [ ]:
from zipfile import ZipFile
bundle = Path(result["zip_path"])
print("ZIP:",bundle)
print("Size (MB):",round(bundle.stat().st_size/1024**2,2))
with ZipFile(bundle) as archive:
    print("Files:",len(archive.namelist()))
    print("CRC check:","PASSED" if archive.testzip() is None else "FAILED")
print("\nSuggested prompt: ")
print((OUTPUT_DIR / "AI_PROMPT.md").read_text(encoding="utf-8")[:1100])


## 10 — FEATURE-SPACE ANALYSIS (opsional, tidak mengubah metrik canonical)

Feature-space hanya ditambahkan bila `outputs/feature_analysis/feature_analysis_manifest.json` tersedia dan audit berhasil. Untuk menjaga exporter v2.1 yang telah teruji, feature-space dikemas sebagai **ZIP tambahan** bersama bundle AI-ready yang sudah dibuat, tanpa mengubah AIA/LTA/FR/BWT existing. Tidak termasuk gambar dataset, embeddings mentah, atau bobot model.

In [ ]:
FEATURE_DIR = PROJECT_ROOT / "outputs" / "feature_analysis"
FEATURE_MANIFEST = FEATURE_DIR / "feature_analysis_manifest.json"
if not FEATURE_MANIFEST.is_file():
    print("FEATURE-SPACE ANALYSIS: belum ada. Jalankan notebook 09 setelah audit checkpoint.")
else:
    feature_status = json.loads(FEATURE_MANIFEST.read_text(encoding="utf-8"))
    print("Validated embedding runs:", feature_status.get("validated_embedded_runs", 0))
    print("Missing/invalid:", feature_status.get("missing_or_invalid", 0))
    print("Fair attention-feature pairs:", feature_status.get("n_fair_feature_attention_pairs", 0))
    for fn in ["inter_class_similarity.csv", "intra_class_dispersion.csv",
               "feature_confusion_join.csv", "feature_correlation_summary.csv",
               "attention_feature_summary.csv", "feature_analysis_errors.csv"]:
        p=FEATURE_DIR/fn
        if p.exists() and p.stat().st_size>0:
            try:
                df = pd.read_csv(p)
                print(fn, "rows:", len(df))
                display(df.head(5))
            except pd.errors.EmptyDataError:
                print(fn, "empty")
    merge_engine = PROJECT_ROOT / "scripts" / "merge_feature_ai_bundle.py"
    if merge_engine.is_file() and result.get("zip_path"):
        import subprocess, sys
        combined_zip = OUTPUT_DIR / "ai_ready_evaluation_with_feature_space.zip"
        subprocess.run([sys.executable, str(merge_engine),
                        "--ai-zip", str(result["zip_path"]),
                        "--feature-dir", str(FEATURE_DIR),
                        "--output", str(combined_zip)], check=True)
        print("Combined evidence ZIP:", combined_zip)
